[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/extendtrajectories.ipynb)

# Extend Conversation Trajectories

*Next user turns that make your model use the whole conversation, in any language.*

## The problem this addresses
In most multi-turn training data, the next user message can be answered from the assistant's last reply. Models which train on these will learn that recent context is enough. In production, that fails in ways like:
- A user asks for something that sounds harmless, but a rule from the system prompt rules it out
- A request the model declined earlier comes back smaller or more urgent, and the model approves it
- A detail the user gave early on should change the answer, but the model never uses it

Public benchmarks such as [MultiChallenge](https://labs.scale.com/leaderboard/multichallenge) and [τ-bench](https://taubench.com/leaderboard?benchmark=core) measure these behaviours, and frontier models still miss some of them.

## What Extend Trajectories does
For each conversation in your dataset, Extend Trajectories writes the user's next message, in the same user's voice and the conversation's own language. A model that answers from the last reply alone is likely to get it wrong, while a model that keeps track of the whole conversation can get it right.

Append the message to the conversation, let the model you are training respond, and you have rollouts that reward looking back instead of answering from the last reply alone.

Your source dataset stays untouched. Each run produces a new extended dataset with the generated message in an `extended_prompt` column.

**Availability: Extend Trajectories is currently available through the Adaptive Data API, not the web UI.**

Resources:
- Documentation: https://docs.adaptionlabs.ai/adaptive-data/extend-trajectories/
- Runbook page: https://docs.adaptionlabs.ai/runbooks/extend-trajectories/

> ⏱ Runtime: minutes · 💳 Cost: low

## Example follow-ups

**Wealth-management chatbot.** A retiree said they could accept at most a 10% loss, and the assistant declined to put \$100,000 into a high-risk fund. Generated next message:
> "That makes sense, but since I don't need the income, could I put \$100,000 into Apex and the rest into Conservative Income?"

**Chinese-language chat.** At the start, the user asked for a 3,500-character plan. Section 3 lists a drone inspection system, and section 5 lists funding sources. Generated next message:
> "现在文档字数达到最初要求的 3500 字了吗？另外，第三部分提到的基于无人机技术的监理巡查系统，能否结合第五部分的科研经费来源，补充说明该项目的预算分配和预期回报周期？"
>
> *(Has the document reached the 3,500 characters I asked for at the start? And can you add a budget split and payback period for the drone inspection system in section 3, using the funding sources in section 5?)*

Each message sounds reasonable on its own. Answering it well depends on something earlier in the conversation.


## Walkthrough

This walkthrough demonstrates how to use Adaptive Data for trajectory extension.

Prerequisites:
- Kindly obtain your API key from the Adaption platform and insert it in the Google Colab notebook's secrets as `ADAPTION_API_KEY`.


In [ ]:
%pip install adaption

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Prepare a dataset

Users can submit a dataset via Hugging Face or local uploads. For our quickstart, we'll use the WildChat conversational dataset from Hugging Face.

In [ ]:
# 2. Load the data you have in mind

########### via HuggingFace
dataset = client.datasets.create(
    source={
        "url": "https://huggingface.co/datasets/allenai/WildChat-1M",
        "files": [
            "data/train-00000-of-00014.parquet"
        ],
    },
)

########### via Local File
# dataset = client.datasets.upload_file("/path/to/file.ext")

dataset_id = dataset.dataset_id
print(dataset_id)

In [ ]:
# 3. Check if the upload is complete
import time

while True:
    status = client.datasets.get_status(dataset_id)
    if status.status == "failed":
        err = status.error_data
        msg = (err and err.message) or "unknown error"
        raise RuntimeError(f"Ingestion failed: {msg}")
    if status.row_count is not None:
        print(f"Imported {status.row_count} rows")
        break
    time.sleep(5)

## Extend Trajectories

Once the data is created, users can extend the trajectories as follows.

In [ ]:
# 4. Estimate the extension.

column_mapping = {
    "chat":"conversation",
}
job_specification = {"max_rows":20}

estimate = client.datasets.extend_trajectories(
    dataset_id=dataset_id,
    column_mapping=column_mapping,
    job_specification=job_specification,
    estimate=True
)
print(f"~{estimate.estimated_minutes:.0f} min, {estimate.estimated_credits_consumed:.0f} credits")

All good? Run it 🏃.

In [ ]:
# 5. Start the extension
extension = client.datasets.extend_trajectories(
    dataset_id=dataset_id,
    column_mapping=column_mapping,
    job_specification=job_specification,
)
print(f"Run started: {extension.run_id}")

In [ ]:
# 6. Await on the datasets
status = client.datasets.wait_for_completion(
    extension.dataset_id,
    timeout=3600
)
if status.status == "failed":
  raise RuntimeError(f"Run {extension.run_id} has failed because of {status.error_data}")
print(status.status)

## Inspect the finished data

There are three ways to view the finished data:
1. Read in bytes
```
client.datasets.download(...).read()
```
2. Write to file
```
client.datasets.download(...).write_to_file(...)
```
3. Read the text
```
client.datasets.download(...).text()
```

We'll use `write_to_file` for the following demonstration.

In [ ]:
# 7. Download and inspect the data
download = client.datasets.download(
    dataset_id=extension.dataset_id,
    file_format="csv"
)

download.write_to_file("wildchat_extended.csv")

In [ ]:
import pandas as pd
import json
import textwrap

data = pd.read_csv("wildchat_extended.csv")
WIDTH = 80

def show(row, limit=500):
    turns = json.loads(row["original_prompt"])
    turns.append({"role": "assistant", "content": row["original_completion"]})

    for i, turn in enumerate(turns, 1):
        header = f"── Turn {i} · {turn['role'].upper()} "
        print(header + "─" * (WIDTH - len(header)))
        print(textwrap.fill(turn["content"][:limit], WIDTH), "\n")

    print("═" * WIDTH)
    print("GENERATED NEXT USER TURN")
    print("═" * WIDTH)
    print(textwrap.fill(row["extended_prompt"], WIDTH))

show(data.iloc[4])


## Using the output
Extend Trajectories writes the question. Your pipeline writes the answer.

- **Training:** append `extended_prompt` as a user turn, let the model you're training respond with the same system prompt and tools, grade the replies, and train on the ones that got it right, with SFT or as RL episodes.
- **Evaluation:** each extended conversation is a ready-made test case. Give your agent the history and the generated turn, and check whether its reply uses what came earlier.

## Tips
- **Choose conversations with something to remember.** Policies, constraints, decisions and amounts give the generator more to build on, and longer conversations give it more.
- **Roll out under the original conditions.** For agents, the environment state should match what already happened in the conversation.
- **Start small.** Run a few hundred rows, check the output, then scale up.

## Learn more
- [Documentation](https://docs.adaptionlabs.ai/adaptive-data/extend-trajectories/)
- [Runbook page](https://docs.adaptionlabs.ai/runbooks/extend-trajectories/)
- [Book a call](https://adaptionlabs.ai/book-a-call)



*Demo data: [WildChat-1M](https://huggingface.co/datasets/allenai/WildChat-1M) (Allen Institute for AI).*
